Create a flattened parquet file ommitting the messages key.

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Point to your newly created enriched dataset
file_path = Path('data/studychat_with_reliance_label.jsonl')
cleaned_file_path = Path("data/clean_analytical_dataset.parquet")

flat_data = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)

        # 1. Safely extract the original Dialogue Act label (if it exists)
        da_label = None
        if 'llm_label' in record and isinstance(record['llm_label'], dict):
            da_label = record['llm_label'].get('label')

        # 2. Safely extract your NEW Reliance Label (if it exists)
        # Note: interactionCount == 0 probably won't have this, so we use .get() to avoid KeyErrors
        rel_label = None
        rel_reason = None
        if 'reliance_label' in record and isinstance(record['reliance_label'], dict):
            rel_label = record['reliance_label'].get('label')
            rel_reason = record['reliance_label'].get('reason')

        # 3. Append only the clean, flat data we actually need for regressions and charts
        flat_data.append({
            'user_id': record.get('userId'),
            'chat_id': record.get('chatId'),
            'semester': record.get('semester'),
            'topic': record.get('topic'),
            'interaction_index': record.get('interactionCount'),
            'dialogue_act': da_label,
            'reliance_category': rel_label,
            # Optional: keep the reason if you want to print examples in your thesis!
            'reliance_reason': rel_reason
        })

# 4. Create the Pandas DataFrame
df = pd.DataFrame(flat_data)

# 5. Clean up: Drop rows that don't have a reliance label (like interaction 0)
df = df.dropna(subset=['reliance_category'])

print(f"Successfully loaded {len(df):,} classified interactions!")
display(df.head())

# Saving in Notebook 01
df.to_parquet('../data/clean_analytical_dataset.parquet', engine='pyarrow')

# Loading in Notebooks 02, 03, 04
df = pd.read_parquet('../data/clean_analytical_dataset.parquet')